In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv langchain-community langchain-core langchain langchain-openai langchain-chroma

In [3]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [4]:
# .env 파일에서 환경 변수 로드 (.env 파일에는 OPENAI API 키값을 적으면 됩니다. -> OPENAI_API_KEY=...)
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")

In [5]:
llm = OpenAI(api_key=api_key)

In [6]:
# <문자열 프롬프트 템플릿>

# 라이브러리 불러오기
from langchain_core.prompts import PromptTemplate
# 주어진 주제에 대한 농담을 요청하는 프롬프트 템플릿 정의
prompt_template = PromptTemplate.from_template("주제 {topic}에 대해 금융 관련 짧은 조언을 해주세요")
# '투자' 주제로 프롬프트 템플릿 호출
prompt_template.invoke({"topic": "투자"})

StringPromptValue(text='주제 투자에 대해 금융 관련 짧은 조언을 해주세요')

In [7]:
# <챗 프롬프트 템플릿>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate

# 챗 프롬프트 템플릿 정의: 사용자와 시스템 간의 메시지를 포함
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 유능한 금융 조언가입니다."),
    ("user", "주제 {topic}에 대해 금융 관련 조언을 해주세요")
])
# '주식' 주제로 챗 프롬프트 템플릿 호출
prompt_template.invoke({"topic": "주식"})

ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='주제 주식에 대해 금융 관련 조언을 해주세요', additional_kwargs={}, response_metadata={})])

In [8]:

# <메시지 자리 표시자 템플릿>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

# (방법1) 메시지 자리 표시자를 포함한 챗 프롬프트 템플릿 정의
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 유능한 금융 조언가입니다."),
    MessagesPlaceholder("msgs")
])
# 메시지 리스트를 'msgs' 자리 표시자에 전달하여 호출
prompt_template.invoke({"msgs": [HumanMessage(content="안녕하세요!")]})

ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={})])

In [9]:
# (방법2) MessagesPlaceholder 클래스를 명시적으로 사용하지 않고도 비슷한 작업을 수행할 수 있는 방법
prompt_template = ChatPromptTemplate.from_messages([
("system", "당신은 유능한 금융 조언가입니다."),
("placeholder", "{msgs}") # <-- 여기서 'msgs'가 자리 표시자로 사용됩니다.
])
# 메시지 리스트를 'msgs' 자리 표시자에 전달하여 호출
prompt_template.invoke({"msgs": [HumanMessage(content="안녕하세요!")]})

ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={})])

In [10]:
# <`PromptTemplate`를 이용한 퓨샷 프롬프트>

# 라이브러리 불러오기
from langchain_core.prompts import PromptTemplate
# 질문과 답변을 포맷하는 프롬프트 템플릿 정의
example_prompt = PromptTemplate.from_template("질문: {question}\n답변: {answer}")

examples = [
    {
        "question": "주식 투자와 예금 중 어느 것이 더 수익률이 높은가?",
        "answer": """
후속 질문이 필요한가요: 네.
후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
후속 질문: 예금의 평균 이자율은 얼마인가요?
중간 답변: 예금의 평균 이자율은 연 1%입니다.
따라서 최종 답변은: 주식 투자
""",
    } ,
    {
        "question": "부동산과 채권 중 어느 것이 더 안정적인 투자처인가?",
        "answer": """
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
후속 질문: 채권의 위험도는 어느 정도인가요?
중간 답변: 채권의 위험도는 낮은 편입니다.
따라서 최종 답변은: 채권
""",
    },
]

print(example_prompt.invoke(examples[0]).to_string())

질문: 주식 투자와 예금 중 어느 것이 더 수익률이 높은가?
답변: 
후속 질문이 필요한가요: 네.
후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
후속 질문: 예금의 평균 이자율은 얼마인가요?
중간 답변: 예금의 평균 이자율은 연 1%입니다.
따라서 최종 답변은: 주식 투자



In [11]:
# <`FewShotPromptTemplate` 를 이용한 퓨샷 프롬프트>

# 라이브러리 불러오기
from langchain_core.prompts import FewShotPromptTemplate
# FewShotPromptTemplate 생성
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="질문: {input}",
    input_variables=["input"],
)
# '부동산 투자' 주제로 프롬프트 호출 및 출력
print(
    prompt.invoke({"input": "부동산 투자의 장점은 무엇인가?"}).to_string()
)

질문: 주식 투자와 예금 중 어느 것이 더 수익률이 높은가?
답변: 
후속 질문이 필요한가요: 네.
후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
후속 질문: 예금의 평균 이자율은 얼마인가요?
중간 답변: 예금의 평균 이자율은 연 1%입니다.
따라서 최종 답변은: 주식 투자


질문: 부동산과 채권 중 어느 것이 더 안정적인 투자처인가?
답변: 
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
후속 질문: 채권의 위험도는 어느 정도인가요?
중간 답변: 채권의 위험도는 낮은 편입니다.
따라서 최종 답변은: 채권


질문: 부동산 투자의 장점은 무엇인가?


In [12]:
# <예제 선택기를 이용한 퓨샷 프롬프트>

# 라이브러리 불러오기
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings

# 예제 선택기 초기화
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,  # 사용할 예제 목록
    OpenAIEmbeddings(api_key=api_key),  # 임베딩 생성에 사용되는 클래스
    Chroma,  # 임베딩을 저장하고 유사도 검색을 수행하는 벡터 스토어 클래스
    k=1,  # 선택할 예제의 수
)

# 입력과 가장 유사한 예제 선택
question = "부동산 투자의 장점은 무엇인가?"
selected_examples = example_selector.select_examples({"question": question})

# 선택된 예제 출력
print(f"입력과 가장 유사한 예제: {question}")
for example in selected_examples:
    print("\n")
    for k, v in example.items():
        print(f"{k}: {v}")

입력과 가장 유사한 예제: 부동산 투자의 장점은 무엇인가?


question: 부동산과 채권 중 어느 것이 더 안정적인 투자처인가?
answer: 
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
후속 질문: 채권의 위험도는 어느 정도인가요?
중간 답변: 채권의 위험도는 낮은 편입니다.
따라서 최종 답변은: 채권



In [13]:
# <퓨샷 프롬프트 AI 모델 적용>

# 라이브러리 불러오기
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI

# 예제 프롬프트 템플릿 생성
example_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="질문: {question}\n답변: {answer}"
)

# 퓨샷 프롬프트 템플릿 설정
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="다음은 금융 관련 질문과 답변의 예시입니다:",
    suffix="질문: {input}\n답변:",
    input_variables=["input"]
)

In [14]:
# AI 모델 설정
model = ChatOpenAI(model_name="gpt-4o")

In [15]:
# 체인 구성 및 실행
chain = prompt | model  # RunnableSequence를 사용하여 체인 연결

response = chain.invoke({"input": "부동산 투자의 장점은 무엇인가?"})  # invoke 메서드 사용
print(response.content)

부동산 투자의 장점은 여러 가지가 있습니다. 첫째, 부동산은 물리적 자산이기 때문에 인플레이션에 대한 방어 수단이 될 수 있습니다. 둘째, 부동산은 임대 수익을 통해 정기적인 현금 흐름을 제공할 수 있습니다. 셋째, 시간이 지나면서 가치가 상승할 가능성이 있어 장기적인 자본 이득을 기대할 수 있습니다. 넷째, 다양한 투자 전략을 통해 포트폴리오 다각화에 기여할 수 있습니다. 마지막으로, 세제 혜택이나 정부 정책의 지원을 받을 수 있는 경우도 있습니다. 이러한 장점들 때문에 많은 투자자들이 부동산을 매력적인 투자처로 고려하게 됩니다.


랭체인 1.0에서는 `from langchain import hub`가 동작하지 않습니다. 따라서 `pip install langchain-classic`로 `langchain-classic`를 설치하고 `from langchain_classic import hub`를 사용하세요.

In [23]:
# <랭체인 허브의 특정 프롬프트 불러오기>
# pip install langchain-classic
# from langchain_classic import hub
from langchainhub import Client

# 1. 클라이언트 초기화
client = Client()

# 최신 버전의 프롬프트 불러오기
prompt = client.pull("hardkothari/prompt-maker")
# 특정 버전의 프롬프트 불러오기
client.pull("hardkothari/prompt-maker")

C:\Users\an9383\AppData\Local\Temp\ipykernel_1636\2139355067.py:10: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull("hardkothari/prompt-maker")
d:\WorkSpace\Python\langchain-tutorial\Ch01. Langchain Basics\venv\lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)
C:\Users\an9383\AppData\Local\Temp\ipykernel_1636\2139355067.py:12: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  client.pull("hardkothari/prompt-maker")
d:\WorkSpace\Python\langchain-tutorial\Ch01. Langchain Basics\venv\lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk`

'{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "SystemMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are an expert Prompt Writer for Large Language Models.\\n\\n", "input_variables": [], "template_format": "f-string"}}}}, {"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "Your goal is to improve the prompt given below for {task} :\\n--------------------\\n\\nPrompt: {lazy_prompt}\\n\\n--------------------\\n\\nHere are several tips on writing great prompts:\\n\\n-------\\n\\nStart the prompt by stating that it is an expert in the subject.\